In [1]:
import os
import sys
import socket

import pandas as pd
import numpy as np
import dask
import dask.dataframe as dd
import dask.array as da
import dask.bag as db
from dask_sql import Context
from dask_jobqueue import SGECluster
from dask.distributed import Client, LocalCluster

/usr/local/icsoftware/anaconda3/envs/jupyterhub_base_v1.5/lib/python3.7/site-packages/dask_jobqueue/core.py:19: FutureWarning: format_bytes is deprecated and will be removed in a future release. Please use dask.utils.format_bytes instead.
  from distributed.utils import format_bytes, parse_bytes, tmpfile
/usr/local/icsoftware/anaconda3/envs/jupyterhub_base_v1.5/lib/python3.7/site-packages/dask_jobqueue/core.py:19: FutureWarning: parse_bytes is deprecated and will be removed in a future release. Please use dask.utils.parse_bytes instead.
  from distributed.utils import format_bytes, parse_bytes, tmpfile
/usr/local/icsoftware/anaconda3/envs/jupyterhub_base_v1.5/lib/python3.7/site-packages/dask_jobqueue/core.py:19: FutureWarning: tmpfile is deprecated and will be removed in a future release. Please use dask.utils.tmpfile instead.
  from distributed.utils import format_bytes, parse_bytes, tmpfile
/usr/local/icsoftware/anaconda3/envs/jupyterhub_base_v1.5/lib/python3.7/site-packages/dask_job

In [2]:
#Get DeID data
cluster = LocalCluster(n_workers=8, memory_limit='128gb')
client = Client(cluster)

rwd_output = './assets/data/'

def load_register_table(data_asset, table, **kwargs):
    return dd.read_parquet(f'/wynton/protected/project/ic/data/parquet/{data_asset}/{table}/', **kwargs)

client.dashboard_link

note_meta = load_register_table("DEID_CDW", "note_metadata")
note_text = load_register_table("DEID_CDW", "note_text")


In [3]:
accessionnumbers = ['']
braintumor_meta = {}

for num in accessionnumbers:
    test = note_meta[note_meta['accessionnumber'] == num]
    test = test.compute()
    
    braintumor_meta[num] = test

braintumor_meta

{'885F91772A760':           patientepicid patientdurablekey   deid_note_key    deid_note_id  \
 1088496  D5E4A4ACF3AB8E    DE5A42C324D97C  DCBF132F70BEF4            None   
 308629   D5E4A4ACF3AB8E    DE5A42C324D97C  D445CFBFE02265  D03237EE10BF8F   
 
         deid_note_csn_id procedureorderfactid accessionnumber  \
 1088496   D691EE032A2996       D01B84FD28C123   885F91772A760   
 308629    DBFABC0FF568F5       D01B84FD28C123   885F91772A760   
 
         accessionnumber2 accessionnumber3 encounterfactid  ...  \
 1088496             None             None  DF2EF1A2FCF5D5  ...   
 308629              None             None            None  ...   
 
         note_type_noadd_c from_order_narr_impr      encounter_type  \
 1088496              None     order_impression  Hospital Encounter   
 308629               None                  HNO                None   
 
         enc_dept_name  enc_dept_specialty  employeeepicid providerepicid  \
 1088496  RAD ONC PARN  Radiation Oncology  D14BC1DC

In [42]:
braintumor_meta['']['note_type']

2896157    Imaging
584844     Imaging
Name: note_type, dtype: object

In [4]:
from collections import defaultdict
clinical_notes = defaultdict(list)

for accessionnum, entries in braintumor_meta.items():
    for _, entry in entries.iterrows():
        currow = dict(entry.items())
        notes = note_text[note_text['deid_note_key'] == currow['deid_note_key']]
        notes = notes.compute()
        
        clinical_notes[accessionnum].append(dict(notes)['note_text'].iloc[0])

In [5]:
print(clinical_notes)

defaultdict(<class 'list'>, {'885F91772A760': ['\n TRIGEMINAL NEURALGIA\n ***** *****: 11/15/2004.\n \n CLINICAL HISTORY: Trigeminal neuralgia.\n \n COMPARISON: None.\n \n TECHNIQUE:\n \n The following sequences focusing on the midbrain and pons were\n obtained at 1.5 *****: Sagittal T1 postcontrast, axial 3D SPGR\n postcontrast, 3D fast spin-echo T2 with fat saturation.\n \n FINDINGS:\n \n On these limited images of the brain, vascular structures are noted\n bilaterally near the root entry zone of the 5th nerve.\n \n There is right-sided mastoid air cell fluid.\n \n The limited visualization of the brain and orbits are normal.\n \n IMPRESSION:\n \n 1. On this limited examination focusing on the pons and midbrain,\n prominent vascular structures are noted bilaterally near the root\n entry zone of the 5th cranial nerve.\n \n 2. Right mastoid air cell fluid which may represent a\n postobstructive condition. There is no significant enhancement.\n \n END OF IMPRESSION:\n \n \n', 'IMPRESSIO

In [26]:
#This cell holds the code for the function that gets rid of the python tag 
import re

def remove_tag(response):
    match = re.search(r'```python\n(.)+```', response, re.DOTALL)
    
    if match:
        return match.group(1).strip()
    else:
        return None

In [32]:
from openai import AzureOpenAI

#creating the dictioanry that will hold the question answer pairs
QAPairs = defaultdict(dict)

#Create the client that will be used to 
client = AzureOpenAI(api_key="",
                    api_version="2025-04-01-preview",
                    azure_endpoint="https://unified-api.ucsf.edu/general")

for accession, texts in clinical_notes.items():
    
    #Comebine all of the text
    all_text = ""
    for idx, text in enumerate(texts):
        all_text += 'Clinical Note (' + str(idx + 1) + ') :' + text + '\n\n'

    #Chat creation for making question answer pairs
    complete_llm = client.chat.completions.create(
            model="gpt-4o-2024-11-20", # Or another suitable model
            messages=[
                {"role": "system", "content": "You are a helpful assistant. You will be given the clinical notes \
                of a patient and will create 20 question answer pairs about the brain tumor patient.\
                Please return this as a python list and only a python list."},
                {"role": "user", "content": all_text},
            ]
        )

    partial_llm = client.chat.completions.create(
            model="gpt-4o-2024-11-20", # Or another suitable model
            messages=[
                {"role": "system", "content": "You are a helpful assistant. You will be given the clinical notes \
                of a patient. Please answer the following questions listed below\
                and also create more question answer pairs from these clinical notes (20 questions in total). \
                Please return this as a python list and only a python list. \
                \
                QUESTIONS: \
                 - Is there a tumor present based on the reports? \
                 - Where is the location of the tumor in ? \
                 - What are the signal characteristics on T1, T2, and FLAIR? \
                 - Is there evidence of necrosis or hemorrhage inside the tumor? \
                 - Are new lesions or metastases present compared to prior exams? \
                 - Is this change consistent with true progression or pseudoprogression? \
                 - Is this change consistent with treatment effect (e.g., radiation necrosis)? \
                 - Is this patient eligible for surgery, biopsy, or stereotactic radiosurgery based on lesion size and location?"},
                {"role": "user", "content": all_text},
            ]
        )


    categorized = client.chat.completions.create(
            model="gpt-4o-2024-11-20", # Or another suitable model
            messages=[
                {"role": "system", "content": "You are a helpful assistant. You will be given the clinical notes \
                of a patient and will create 20 question answer pairs about the brain tumor patient.\
                Please also categorize the question answer pairs into categorized based on the question asked.\
                Please return this as a python dictionary and only a python dictionary.\
                 \
                 EX: {'Category A': [(question1, answer 1)], 'Category B': [(question1, answer 1)]}"},
                {"role": "user", "content": all_text},
            ]
        )

    #get rid of python tags from responses
    #complete_llm = remove_tag(complete_llm.choices[0].message.content)
    #partial_llm = remove_tag(partial_llm.choices[0].message.content)
    #categorized = remove_tag(categorized.choices[0].message.content)
    
    #Add all of the QA responses with their respective reports
    QAPairs[accession]['report'] = all_text
    QAPairs[accession]['complete_llm'] = complete_llm.choices[0].message.content
    QAPairs[accession]['partial_llm'] = partial_llm.choices[0].message.content
    QAPairs[accession]['categorized'] = categorized.choices[0].message.content

In [49]:
QAPairs['']['complete_llm']

'```python\n[\n    {"question": "Is there a tumor present based on the reports?", "answer": "No, there is no mention of a tumor in the clinical notes."},\n    {"question": "Where is the location of the tumor?", "answer": "There is no tumor present according to the clinical notes."},\n    {"question": "What are the signal characteristics on T1, T2, and FLAIR?", "answer": "Sagittal T1 postcontrast, axial 3D SPGR postcontrast, and 3D fast spin-echo T2 with fat saturation sequences were performed, but no abnormal signal characteristics were noted."},\n    {"question": "Is there evidence of necrosis or hemorrhage inside the tumor?", "answer": "No tumor is present, so evidence of necrosis or hemorrhage is not applicable."},\n    {"question": "Are new lesions or metastases present compared to prior exams?", "answer": "No comparison was available, and no lesions or metastases were mentioned in the clinical notes."},\n    {"question": "Is this change consistent with true progression or pseudopr

In [51]:


# DO NOT INCLUDE
# questions about comparisons, clinical history, technique
# FOCUS ON Impression and findings

In [39]:
test = """ ```python\n
    {
    'Clinical History': [
        ('What is the clinical history provided in the notes?', 'Trigeminal neuralgia.')
    ],
    'Comparison': [
        ('Was there any comparison made during the examination?', 'No, there was no comparison made.')
    ],
    'Technique Used': [
        ('What technique was used for the examination?', 'Sagittal T1 postcontrast, axial 3D SPGR postcontrast, and 3D fast spin-echo T2 with fat saturation focusing on the midbrain and pons.')
    ],
    'Findings': [
        ('What was observed near the root entry zone of the 5th cranial nerve?', 'Prominent vascular structures were noted bilaterally.')
    ],
    'Additional Findings': [
        ('What was noted on the right side during the examination?', 'Right-sided mastoid air cell fluid was noted.')
    ],
    'Visualization': [
        ('Were the brain and orbits fully visualized during the examination?', 'No, the visualization of the brain and orbits was limited.')
    ],
    'Impression': [
        ('What is the first impression from the examination?', 'Prominent vascular structures are noted bilaterally near the root entry zone of the 5th cranial nerve.')
    ],
    'Right Mastoid Findings': [
        ('What is the second impression from the examination?', 'Right mastoid air cell fluid which may represent a postobstructive condition, with no significant enhancement.')
    ]
}```"""

match = re.search(r'```python\n+(.+)```', test, re.DOTALL)
print(match.group(1).strip())
test = dict(test)
#print(type(test))

{
    'Clinical History': [
        ('What is the clinical history provided in the notes?', 'Trigeminal neuralgia.')
    ],
    'Comparison': [
        ('Was there any comparison made during the examination?', 'No, there was no comparison made.')
    ],
    'Technique Used': [
        ('What technique was used for the examination?', 'Sagittal T1 postcontrast, axial 3D SPGR postcontrast, and 3D fast spin-echo T2 with fat saturation focusing on the midbrain and pons.')
    ],
    'Findings': [
        ('What was observed near the root entry zone of the 5th cranial nerve?', 'Prominent vascular structures were noted bilaterally.')
    ],
    'Additional Findings': [
        ('What was noted on the right side during the examination?', 'Right-sided mastoid air cell fluid was noted.')
    ],
    'Visualization': [
        ('Were the brain and orbits fully visualized during the examination?', 'No, the visualization of the brain and orbits was limited.')
    ],
    'Impression': [
        ('Wha

ValueError: dictionary update sequence element #0 has length 1; 2 is required